### Silver: static GTFS reference entities (agency, routes, trips, stops, calendar, calendar_dates, shapes, stop_times)
##### Conforms Bronze's raw, duplicate-tolerant capture into one deduplicated, typed row per business key, via the hash-diff MERGE: latest Bronze row per (gtfs_mode, key), hash the business columns, update only on a hash change, insert if new.
##### Batch, not streaming -- static GTFS moves at most daily and the volume is small, so a scheduled MERGE after each Bronze static load is simpler than a stream + checkpoint here.
##### Processes both modes (buses' real-time isn't live yet, but its dimension tables are ready for when it is).

In [0]:
%run ../../commonsetup/Setup-Common

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType,
)
from delta.tables import DeltaTable



### 1: Entity registry
##### Per-entity schema, primary key (always prefixed with gtfs_mode -- a route_id can collide across modes), and which columns are GTFS "YYYYMMDD" date strings needing explicit parsing.
##### TODO: not yet diffed against the real downloaded CSV headers -- a non-standard column name comes through NULL rather than failing the entity, worth checking once real files are in. Also duplicates GTFS_ENTITIES from Bronze rather than sharing via Setup-Common; fine for now, fold in once this is metadata-driven.

In [0]:
ENTITY_REGISTRY = {
    "agency": {
        "schema": StructType([
            StructField("agency_id", StringType()),
            StructField("agency_name", StringType()),
            StructField("agency_url", StringType()),
            StructField("agency_timezone", StringType()),
            StructField("agency_lang", StringType()),
            StructField("agency_phone", StringType()),
        ]),
        "key_cols": ["agency_id"],
        "date_cols": [],
    },
    "routes": {
        "schema": StructType([
            StructField("route_id", StringType()),
            StructField("agency_id", StringType()),
            StructField("route_short_name", StringType()),
            StructField("route_long_name", StringType()),
            StructField("route_desc", StringType()),
            StructField("route_type", IntegerType()),
            StructField("route_url", StringType()),
            StructField("route_color", StringType()),
            StructField("route_text_color", StringType()),
        ]),
        "key_cols": ["route_id"],
        "date_cols": [],
    },
    "trips": {
        "schema": StructType([
            StructField("route_id", StringType()),
            StructField("service_id", StringType()),
            StructField("trip_id", StringType()),
            StructField("trip_headsign", StringType()),
            StructField("trip_short_name", StringType()),
            StructField("direction_id", IntegerType()),
            StructField("block_id", StringType()),
            StructField("shape_id", StringType()),
            StructField("wheelchair_accessible", IntegerType()),
            StructField("bikes_allowed", IntegerType()),
        ]),
        "key_cols": ["trip_id"],
        "date_cols": [],
    },
    "stops": {
        "schema": StructType([
            StructField("stop_id", StringType()),
            StructField("stop_code", StringType()),
            StructField("stop_name", StringType()),
            StructField("stop_desc", StringType()),
            StructField("stop_lat", DoubleType()),
            StructField("stop_lon", DoubleType()),
            StructField("zone_id", StringType()),
            StructField("stop_url", StringType()),
            StructField("location_type", IntegerType()),
            StructField("parent_station", StringType()),
            StructField("wheelchair_boarding", IntegerType()),
            StructField("platform_code", StringType()),
        ]),
        "key_cols": ["stop_id"],
        "date_cols": [],
    },
    "calendar": {
        "schema": StructType([
            StructField("service_id", StringType()),
            StructField("monday", IntegerType()),
            StructField("tuesday", IntegerType()),
            StructField("wednesday", IntegerType()),
            StructField("thursday", IntegerType()),
            StructField("friday", IntegerType()),
            StructField("saturday", IntegerType()),
            StructField("sunday", IntegerType()),
            StructField("start_date", StringType()),
            StructField("end_date", StringType()),
        ]),
        "key_cols": ["service_id"],
        "date_cols": ["start_date", "end_date"],
    },
    "calendar_dates": {
        "schema": StructType([
            StructField("service_id", StringType()),
            StructField("date", StringType()),
            StructField("exception_type", IntegerType()),
        ]),
        "key_cols": ["service_id", "date"],
        "date_cols": ["date"],
    },
    "shapes": {
        "schema": StructType([
            StructField("shape_id", StringType()),
            StructField("shape_pt_lat", DoubleType()),
            StructField("shape_pt_lon", DoubleType()),
            StructField("shape_pt_sequence", IntegerType()),
            StructField("shape_dist_traveled", DoubleType()),
        ]),
        "key_cols": ["shape_id", "shape_pt_sequence"],
        "date_cols": [],
    },
    "stop_times": {
        "schema": StructType([
            StructField("trip_id", StringType()),
            # arrival/departure_time can exceed 24:00:00 for trips that run
            # past midnight -- GTFS spec allows this deliberately, so these
            # stay STRING, never cast to a TIME/TIMESTAMP type.
            StructField("arrival_time", StringType()),
            StructField("departure_time", StringType()),
            StructField("stop_id", StringType()),
            StructField("stop_sequence", IntegerType()),
            StructField("stop_headsign", StringType()),
            StructField("pickup_type", IntegerType()),
            StructField("drop_off_type", IntegerType()),
            StructField("shape_dist_traveled", DoubleType()),
        ]),
        "key_cols": ["trip_id", "stop_sequence"],
        "date_cols": [],
    },
}

In [0]:
# Same set as Bronze's CRITICAL_ENTITIES (redefined here, not shared -- see registry note above).
CRITICAL_ENTITIES = {"trips", "stop_times", "calendar", "routes"}



### 2: Hash-diff MERGE, one entity at a time
##### First run for an entity just writes the deduplicated/typed rows directly (nothing to diff against yet); every run after that MERGEs on the row hash.

In [0]:
def upsert_entity_to_silver(entity: str, schema: StructType, key_cols: list, date_cols: list) -> str:
    bronze_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_gtfs_{entity}"
    silver_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_{entity}"

    if not spark.catalog.tableExists(bronze_table):
        print(f"[{entity}] no Bronze table yet -- skipping")
        return "skipped_no_bronze"

    bronze_df = spark.table(bronze_table)

    business_cols = [f.name for f in schema.fields]
    missing_cols = [c for c in business_cols if c not in bronze_df.columns]
    if missing_cols:
        print(f"[{entity}] WARNING: Bronze has no column(s) {missing_cols} -- "
              f"check the actual GTFS header for this entity; these will be NULL in Silver")

    # Silver enforces its own types regardless of Auto Loader's inference;
    # an unparseable value becomes NULL, not a job failure.
    typed_df = bronze_df.select(
        F.col("gtfs_mode"),
        F.col("ingestion_timestamp"),
        *[
            (F.col(f.name).cast(f.dataType).alias(f.name) if f.name in bronze_df.columns
             else F.lit(None).cast(f.dataType).alias(f.name))
            for f in schema.fields
        ],
    )

    # "YYYYMMDD" strings -- a bare cast to DATE would just null them, so parse explicitly.
    for date_col in date_cols:
        typed_df = typed_df.withColumn(date_col, F.to_date(F.col(date_col), "yyyyMMdd"))

    full_key = ["gtfs_mode"] + key_cols

    # Bronze is append-only, so a key can repeat across drops -- latest ingestion wins.
    recency_window = Window.partitionBy(*full_key).orderBy(F.col("ingestion_timestamp").desc())
    latest_df = (
        typed_df
        .withColumn("_rn", F.row_number().over(recency_window))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    hash_cols = [c for c in typed_df.columns if c not in full_key + ["ingestion_timestamp"]]
    latest_df = (
        latest_df
        .withColumn(
            "row_hash",
            F.sha2(
                F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("<null>")) for c in hash_cols]),
                256,
            ),
        )
        .withColumn("silver_updated_at", F.current_timestamp())
    )

    if not spark.catalog.tableExists(silver_table):
        latest_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
        row_count = spark.table(silver_table).count()
        print(f"[{entity}] created {silver_table} ({row_count} rows)")
        return "created"

    target = DeltaTable.forName(spark, silver_table)
    merge_condition = " AND ".join(f"target.{c} = source.{c}" for c in full_key)

    (
        target.alias("target")
        .merge(latest_df.alias("source"), merge_condition)
        .whenMatchedUpdate(
            condition="target.row_hash <> source.row_hash",
            set={c: f"source.{c}" for c in latest_df.columns},
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"[{entity}] merged into {silver_table}")
    return "merged"



### 3: Run for every registered entity
##### Same per-entity isolation as Bronze -- one bad entity shouldn't block the rest.

In [0]:
run_results = {}  # entity -> "created" | "merged" | "skipped_no_bronze" | "failed"

for entity, cfg in ENTITY_REGISTRY.items():
    try:
        run_results[entity] = upsert_entity_to_silver(
            entity, cfg["schema"], cfg["key_cols"], cfg["date_cols"]
        )
    except Exception as e:
        print(f"[{entity}] FAILED: {e}")
        run_results[entity] = "failed"

print("Silver static run results:", run_results)

### 4: Validation + downstream readiness signal
##### Records the outcome for orchestration to check, and fails loudly if a Gold-critical entity didn't make it in this run.

In [0]:
run_log_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_static_run_log"
run_log_df = spark.createDataFrame(
    [(entity, outcome, entity in CRITICAL_ENTITIES) for entity, outcome in run_results.items()],
    schema="entity STRING, outcome STRING, is_critical BOOLEAN",
).withColumn("run_timestamp", F.current_timestamp())
run_log_df.write.format("delta").mode("append").saveAsTable(run_log_table)

failed_or_missing = {e for e, o in run_results.items() if o in ("failed", "skipped_no_bronze")}
failed_critical = {e for e in failed_or_missing if e in CRITICAL_ENTITIES}

if failed_critical:
    raise RuntimeError(
        f"Critical entities not in Silver after this run: {sorted(failed_critical)}. "
        f"Not safe to trigger Gold's enriched vehicle-position join until these succeed."
    )
elif failed_or_missing:
    print(f"Non-critical entities missing/failed this run: {sorted(failed_or_missing)}. "
          f"Proceeding -- Gold can still build on the entities that succeeded.")
else:
    print("All registered entities merged into Silver successfully. Safe to trigger Gold.")



In [0]:
#%sql
#select * from silver_gtfs_routes limit 20;